# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is published as a Croissant schema and is accessible via the following URL:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the metadata as a dict for easier display
metadata = json.loads(dataset.metadata.to_json())
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets and their fields using Croissant `@id`s.
We will identify all the record sets, fields, and columns and reference them by their `@id`s.


In [ ]:
# List all record sets by @id
print("Available Record Sets (by @id and fields):\n")
record_sets = []
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set.id}")
    field_ids = [field.id for field in record_set.fields]
    print(f"  Fields: {field_ids}")
    record_sets.append(record_set.id)

if not record_sets:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load one or more record sets into DataFrames for analysis. 
Use the record set and field `@id`s identified above.

Below, we extract all available record sets.

In [ ]:
# Extract all available record sets and load into DataFrames
dataframes = {}
if record_sets:
    for record_set_id in record_sets:
        # Use generator to build list of records from record set by @id
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}")

    # Show the columns of the first record set
    first_record_set_id = record_sets[0]
    print(f"\nColumns in first record set ({first_record_set_id}):")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
We will apply common data processing techniques such as filtering on a numeric field, normalizing its values, and grouping the data by a categorical field.

Please identify a numeric field and a group field by referring to their `@id`. The IDs and columns were already listed above.

In this example, we select the first record set and use its columns for further analysis.

In [ ]:
# Select a record set and its columns
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    print(f"Available columns: {df.columns.tolist()}")
    
    # For demonstration, select the first column with numeric data for EDA
    # You can replace 'log_likelihood' or similar below with the field @id that is numeric
    numeric_field = None
    for col in df.columns:
        # Attempt to guess numeric field by dtype or column name
        if pd.api.types.is_numeric_dtype(df[col]) or ('log' in col.lower()):
            numeric_field = col
            break
    
    if numeric_field is not None:
        print(f"Using numeric field for EDA: {numeric_field}")
        # Threshold for filtering records
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Choose a group field distinct from the numeric one (if available)
        group_field = None
        for col in df.columns:
            if col != numeric_field and (df[col].dtype == object or df[col].dtype.name == 'category'):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped by field: {group_field}")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Let's visualize the distribution of the numeric field and, if possible, the grouped averages.

In [ ]:
import matplotlib.pyplot as plt

if record_sets and numeric_field is not None:
    plt.figure(figsize=(8,5))
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()
    
    if 'group_field' in locals() and group_field:
        grouped = df.groupby(group_field)[numeric_field].mean()
        grouped.plot(kind='bar', figsize=(8,4))
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to load and explore a Croissant-structured dataset using the `mlcroissant` library. 

We:
- Discovered the available record sets and fields by their Croissant `@id`s.
- Loaded records into pandas DataFrames.
- Performed basic EDA, filtering, normalization, and grouping.
- Visualized data distributions and grouped metrics.

For more advanced analysis, consider using domain knowledge to select meaningful features and summary statistics.